In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
churn_df=pd.read_csv("../data/teleco_customer_churn_train.csv")

In [3]:
churn_df.shape

(7043, 21)

In [4]:
# For removing extra spaces before and after
churn_df = churn_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [5]:
for col in churn_df.columns:
    if churn_df[col].dtype == "object":
        converted = pd.to_numeric(churn_df[col], errors="coerce")

        # convert if most values are numeric
        if converted.notna().sum() / len(churn_df) > 0.8:
            churn_df[col] = converted

In [6]:
# Removes or updates null values
for col in churn_df.columns:

    missing_ratio = churn_df[col].isnull().mean()

    if missing_ratio < 0.025:
        churn_df = churn_df[churn_df[col].notnull()]
    
    else:
        if churn_df[col].dtype in ["int64","float64"]:
            churn_df[col] = churn_df[col].fillna(churn_df[col].median())
        else:
            churn_df[col] = churn_df[col].fillna(churn_df[col].mode()[0])

In [7]:
churn_df.shape

(7032, 21)

In [8]:
target_col = "Churn"
cols_to_drop = []
for col in churn_df.columns:
    
    if col != target_col:
        unique_ratio = churn_df[col].nunique() / len(churn_df)
        
        if unique_ratio > 0.95:
            cols_to_drop.append(col)
            
churn_df = churn_df.drop(columns=cols_to_drop)

In [9]:
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 
 17  

In [10]:
numeric_cols=churn_df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols=churn_df.select_dtypes(include=["object"]).columns
print("Number of numeric_cols:",len(numeric_cols))
print("Number of categorical_cols:",len(categorical_cols))

Number of numeric_cols: 4
Number of categorical_cols: 16


In [11]:
for col in categorical_cols:
    print(f"{col}: {churn_df[col].unique()}")

gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['No' 'Yes' 'No internet service']
StreamingTV: ['No' 'Yes' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn: ['No' 'Yes']


In [12]:
target_column="Churn"
inputs = churn_df.drop(columns=[target_column])
target = churn_df[target_column]

In [13]:
numeric_cols=inputs.select_dtypes(include=["int64", "float64"]).columns
categorical_cols=inputs.select_dtypes(include=["object"]).columns
print("Number of numeric_cols:",len(numeric_cols))
print("Number of categorical_cols:",len(categorical_cols))

Number of numeric_cols: 4
Number of categorical_cols: 15


In [14]:
numeric_cols

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [15]:
binary_cols = [
    col for col in categorical_cols
    if inputs[col].nunique() == 2
]
for col in binary_cols:
    unique_vals = churn_df[col].dropna().unique()
    mapping = {
        unique_vals[0]: 0,
        unique_vals[1]: 1
    }
    inputs[col] = inputs[col].map(mapping)

In [16]:
multiple_cols = [
    col for col in categorical_cols
    if inputs[col].nunique() > 2
]

In [17]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_dfs = []

for col in multiple_cols:

    encoded = encoder.fit_transform(inputs[[col]])

    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out([col]),
        index=inputs.index
    )

    encoded_dfs.append(encoded_df)

# combine all encoded columns
encoded_all = pd.concat(encoded_dfs, axis=1)

# drop original columns
inputs = inputs.drop(columns=multiple_cols)

# add encoded columns
inputs = pd.concat([inputs, encoded_all], axis=1)

In [18]:
inputs

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,0,0,1,0,0,29.85,29.85,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,1,0,34,1,1,56.95,1889.50,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1,0,1,0,2,1,0,53.85,108.15,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,1,0,45,0,1,42.30,1840.75,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0,0,1,0,2,1,0,70.70,151.65,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,0,1,24,1,0,84.80,1990.50,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7039,0,0,0,1,72,1,0,103.20,7362.90,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
7040,0,0,0,1,11,0,0,29.60,346.45,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
7041,1,1,0,0,4,1,0,74.40,306.60,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [19]:
!pip install pyarrow --quiet

In [20]:
target_df = pd.DataFrame(target)
target_df.to_parquet("target.parquet")

In [21]:
target_df = pd.DataFrame(inputs)
target_df.to_parquet("inputs.parquet")